#### How to create dataframe using dictionary?
- **How to access dictionary elements?**

- Useful when data comes as **JSON-like objects**.
- **Column order** is **not guaranteed**.
- **Schema** is **inferred automatically**.

**Spark rule for createDataFrame**

| Data Row Type   |     How Spark Interprets            |
| --------------- | ----------------------------------- |
| `Tuple / List`  | `Each element` → `one column`       |
| `Dictionary`    | `Keys → columns`                    |

In [0]:
data = [{'Name':'Jayanth', 'ID':7576, 'Country':'USA', 'Status': True, 'Ratio': 10.5, "long":9988770011},
        {'Name':'Rupesh', 'ID':9800, 'Country':'USA', 'Status': False, 'Ratio': 16.3, "long":9688673011},
        {'Name':'Thrusanth', 'ID':6543, 'Country':'IND', 'Status': True, 'Ratio': 12.1, "long":9788770011},
        {'Name':'Jahangeer', 'ID':5321, 'Country':'USA', 'Status': False, 'Ratio': 18.8, "long":9482450911},
        {'Name':'Sowmya', 'ID':4956, 'Country':'INA', 'Status': True, 'Ratio': 19.4, "long":9346772211}]

df_dict_dtype = spark.createDataFrame(data)
display(df_dict_dtype)

Country,ID,Name,Ratio,Status,long
USA,7576,Jayanth,10.5,true,9988770011
USA,9800,Rupesh,16.3,false,9688673011
IND,6543,Thrusanth,12.1,true,9788770011
USA,5321,Jahangeer,18.8,false,9482450911
INA,4956,Sowmya,19.4,true,9346772211


In [0]:
dataDictionary = [
        ('James', {'hair':'black','eye':'brown'}),
        ('Michael', {'hair':'brown','eye':None}),
        ('Robert', {'hair':'red','eye':'black'}),
        ('Washington', {'hair':'grey','eye':'grey'}),
        ('Jefferson', {'hair':'brown','eye':''})
        ]

df = spark.createDataFrame(data=dataDictionary, schema = ['name','properties'])
df.printSchema()
display(df)

root
 |-- name: string (nullable = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



name,properties
James,"Map(hair -> black, eye -> brown)"
Michael,"Map(hair -> brown, eye -> null)"
Robert,"Map(hair -> red, eye -> black)"
Washington,"Map(hair -> grey, eye -> grey)"
Jefferson,"Map(hair -> brown, eye -> )"


In [0]:
data = [({'Name':'Jayanth', 'ID':7576, 'Country':'USA', 'Status': True, 'Ratio': 10.5, "long":9988770011},),
        ({'Name':'Rupesh', 'ID':9800, 'Country':'USA', 'Status': False, 'Ratio': 16.3, "long":9688673011},),
        ({'Name':'Thrusanth', 'ID':6543, 'Country':'IND', 'Status': True, 'Ratio': 12.1, "long":9788770011},),
        ({'Name':'Jahangeer', 'ID':5321, 'Country':'USA', 'Status': False, 'Ratio': 18.8, "long":9482450911},),
        ({'Name':'Sowmya', 'ID':4956, 'Country':'INA', 'Status': True, 'Ratio': 19.4, "long":9346772211},)]

df_dict = spark.createDataFrame(data, schema = ['properties'])
display(df_dict)

properties
"Map(ID -> 7576, Name -> Jayanth, Country -> USA, Ratio -> 10.5, long -> 9988770011, Status -> true)"
"Map(ID -> 9800, Name -> Rupesh, Country -> USA, Ratio -> 16.3, long -> 9688673011, Status -> false)"
"Map(ID -> 6543, Name -> Thrusanth, Country -> IND, Ratio -> 12.1, long -> 9788770011, Status -> true)"
"Map(ID -> 5321, Name -> Jahangeer, Country -> USA, Ratio -> 18.8, long -> 9482450911, Status -> false)"
"Map(ID -> 4956, Name -> Sowmya, Country -> INA, Ratio -> 19.4, long -> 9346772211, Status -> true)"


- A `dictionary is iterable`, so Spark treats it like:

      ('Name', 'ID', 'Country', 'Status', 'Ratio', 'long')

- A `dictionary alone` is treated as `multiple fields` unless wrapped inside a `tuple`.

In [0]:
# Using StructType schema
from pyspark.sql.types import StructField, StructType, StringType, MapType,IntegerType

schema = StructType([
    StructField('name', StringType(), True),
    StructField('properties', MapType(StringType(),StringType()),True)
])

df2 = spark.createDataFrame(data=dataDictionary, schema = schema)
df2.printSchema()
display(df2)

root
 |-- name: string (nullable = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



name,properties
James,"Map(hair -> black, eye -> brown)"
Michael,"Map(hair -> brown, eye -> null)"
Robert,"Map(hair -> red, eye -> black)"
Washington,"Map(hair -> grey, eye -> grey)"
Jefferson,"Map(hair -> brown, eye -> )"


In [0]:
df.withColumn("hair", df.properties.getItem("hair")) \
  .withColumn("eye", df.properties.getItem("eye")) \
  .drop("properties") \
  .display()

name,hair,eye
James,black,brown
Michael,brown,null
Robert,red,black
Washington,grey,grey
Jefferson,brown,


In [0]:
df.withColumn("hair", df.properties["hair"]) \
  .withColumn("eye", df.properties["eye"]) \
  .drop("properties") \
  .display()

name,hair,eye
James,black,brown
Michael,brown,null
Robert,red,black
Washington,grey,grey
Jefferson,brown,
